# MMTFv3 Temporal Hardening

Audits branch timing for leakage and runs two deliberate perturbation checks:

- technical branch shifted one bar older
- sequential VPIN branch rebuilt from a cutoff 15 minutes earlier

Use this after training to verify the saved checkpoint is not benefiting from accidental forward-looking alignment.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

TICKERS = ['CL', 'GC']
TUNE_BACKBONE = True
BACKBONE = 'mamba'
USE_PTP = True
USE_FUSED_SPATIAL = True
SPATIAL_ENCODER = 'fused' if USE_FUSED_SPATIAL else 'separate'
SPATIAL_LOOKBACK_BARS = 8
BAR_MINUTES = 5
TARGET_HORIZON_MINUTES = 60
SAMPLE_SESSION = 'usa'
SAMPLE_STRIDE = 12
AE_WINDOW = 21
VAL_RATIO = 0.2
EVAL_MAX_SAMPLES = 4096
SEQ_SHIFT_MINUTES = 15

IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/features')
    RESULTS_PATH = Path('/content/drive/MyDrive/results/mmtfv3_cl_gc')
else:
    DATA_ROOT = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/mmtfv3_cl_gc')

bb_tag = 'tuned' if TUNE_BACKBONE else BACKBONE
ptp_tag = '_ptp' if USE_PTP else ''
prefix = f"{'_'.join(TICKERS)}_mmtfv3_{bb_tag}{ptp_tag}_optuna"
print(prefix)


In [ ]:
import json
import torch
from torch.utils.data import DataLoader

from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    v3_collate_fn,
    unpack_v3_batch,
    SessionSpec,
)
from CTAFlow.models.deep_learning.multi_branch.tft import (
    MMTFv3Core,
    StatefulMMTFv3Core,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
params_path = RESULTS_PATH / f'{prefix}_best_params.json'
with open(params_path) as f:
    best = json.load(f)

prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec('USA', '08:30', '16:00')],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)
dims = prep.get_dims()
FUSED_SPATIAL_CHANNELS = dims['fused_spatial_channels']
FUSED_SPATIAL_BINS = dims['fused_spatial_bins']

base_model = MMTFv3Core(
    f_tech=dims['f_tech'],
    f_seq=dims['f_seq'],
    f_ae=4,
    ae_type=best.get('ae_type', 'vae'),
    d_latent=best['d_latent'],
    d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'],
    recon_weight=best['recon_weight'],
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'],
    d_static_emb=best['d_static_emb'],
    backbone=best.get('backbone', BACKBONE),
    n_heads=best['n_heads'],
    n_layers=best['n_layers'],
    d_ff=best.get('d_ff', 512),
    d_state=best.get('d_state', 16),
    d_conv=best.get('d_conv', 4),
    expand=best.get('expand', 2),
    dropout=best['dropout'],
    grn_dropout=best['grn_dropout'],
    numbars_channels=dims['numbars_channels'],
    vpin_channels=dims['vpin_channels'],
    vpin_bins=dims['vpin_bins'],
    vpin_time=dims['vpin_time'],
    spatial_encoder=SPATIAL_ENCODER,
)
model = StatefulMMTFv3Core(
    base_model=base_model,
    n_tickers=prep.n_tickers,
    quantile_head=USE_PTP,
    ptp_temperature=best.get('ptp_temperature', 1.5),
    state_hidden_dim=best.get('state_hidden_dim', 16),
    state_momentum=best.get('state_momentum', 0.9),
    update_on_eval=True,
).to(device)

ckpt_path = RESULTS_PATH / f'{prefix}_best_model.pth'
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()
print('checkpoint:', ckpt_path)


In [ ]:
tech_lookback = int(best['tech_lookback'])
seq_lookback = int(best['seq_lookback'])
numbars_lookback = int(best.get('numbars_lookback', SPATIAL_LOOKBACK_BARS))

all_samples = prep.build_samples(
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    numbars_lookback=numbars_lookback,
    use_fused_spatial=USE_FUSED_SPATIAL,
    stride=SAMPLE_STRIDE,
    sample_session=SAMPLE_SESSION,
)
split_idx = int(len(all_samples) * (1 - VAL_RATIO))
oos_samples = all_samples[split_idx:]
print('all samples:', len(all_samples))
print('oos samples:', len(oos_samples))
print('sample metadata keys:', sorted(k for k in oos_samples[0].keys() if 'ts' in k or 'date' in k))


In [ ]:
def audit_sample_timing(sample):
    ticker = sample['ticker']
    anchor_ts = pd.Timestamp(sample['anchor_ts'])
    target_end_ts = pd.Timestamp(sample['target_end_ts'])
    ae_end_date = pd.Timestamp(sample['ae_window_end_date']).date()

    df = prep._tech_dfs[ticker]
    bar_loc = df.index.get_loc(anchor_ts)
    tech_max_ts = pd.Timestamp(df.index[bar_loc - 1])

    seq_df = prep._seq_vpin.get(ticker, pd.DataFrame())
    seq_prior = seq_df.index[seq_df.index < anchor_ts] if not seq_df.empty else pd.DatetimeIndex([])
    seq_max_ts = pd.Timestamp(seq_prior[-1]) if len(seq_prior) else None

    nb_ts, _ = prep._numbars_ts.get(ticker, (np.array([], dtype='datetime64[ns]'), None))
    nb_prior = nb_ts[nb_ts < np.datetime64(anchor_ts, 'ns')]
    nb_max_ts = pd.Timestamp(nb_prior[-1]) if len(nb_prior) else None

    close = df['Close'].astype(float)
    expected_target = float(np.log(close.loc[target_end_ts]) - np.log(close.loc[anchor_ts]))

    print('ticker           :', ticker)
    print('anchor_ts        :', anchor_ts)
    print('tech max ts      :', tech_max_ts)
    print('seq max ts       :', seq_max_ts)
    print('numbars max ts   :', nb_max_ts)
    print('ae window end    :', ae_end_date)
    print('target end ts    :', target_end_ts)
    print('sample target    :', float(sample['target']))
    print('expected target  :', expected_target)

    assert tech_max_ts < anchor_ts
    assert seq_max_ts is None or seq_max_ts < anchor_ts
    assert nb_max_ts is None or nb_max_ts < anchor_ts
    assert ae_end_date < anchor_ts.date()
    assert target_end_ts > anchor_ts
    assert np.isclose(float(sample['target']), expected_target, atol=1e-8)

audit_sample_timing(oos_samples[len(oos_samples) // 2])


In [ ]:
SEQ_STORE = {}
for ticker in TICKERS:
    seq_df = prep._seq_vpin.get(ticker, pd.DataFrame())
    if seq_df.empty:
        SEQ_STORE[ticker] = (np.array([], dtype='datetime64[ns]'), np.empty((0, dims['f_seq']), dtype=np.float32))
    else:
        SEQ_STORE[ticker] = (seq_df.index.values, seq_df.values.astype(np.float32))

def clone_sample(sample):
    cloned = dict(sample)
    for key in ('tech_features', 'seq_vpin', 'ae_input'):
        cloned[key] = np.asarray(sample[key], dtype=np.float32).copy()
    if 'fused_spatial' in sample:
        cloned['fused_spatial'] = np.asarray(sample['fused_spatial'], dtype=np.float32).copy()
    if 'numbars_recent' in sample:
        cloned['numbars_recent'] = np.asarray(sample['numbars_recent'], dtype=np.float32).copy()
    if 'vpin_raster_recent' in sample and sample['vpin_raster_recent'] is not None:
        cloned['vpin_raster_recent'] = np.asarray(sample['vpin_raster_recent'], dtype=np.float32).copy()
    return cloned

def perturb_tech_shift_one(sample):
    out = clone_sample(sample)
    tech = out['tech_features']
    shifted = np.zeros_like(tech)
    shifted[1:] = tech[:-1]
    out['tech_features'] = shifted
    return out

def perturb_seq_cutoff_earlier(sample, minutes=SEQ_SHIFT_MINUTES):
    out = clone_sample(sample)
    anchor_ts = pd.Timestamp(sample['anchor_ts']) - pd.Timedelta(minutes=minutes)
    seq_ts, seq_vals = SEQ_STORE[sample['ticker']]
    cut = np.searchsorted(seq_ts, np.datetime64(anchor_ts, 'ns'), side='left')
    if cut > 0:
        lo = max(0, cut - seq_lookback)
        seq_data = seq_vals[lo:cut]
    else:
        f_seq = seq_vals.shape[1] if seq_vals.ndim == 2 and seq_vals.shape[1] > 0 else dims['f_seq']
        seq_data = np.zeros((1, f_seq), dtype=np.float32)
    out['seq_vpin'] = seq_data.astype(np.float32)
    out['seq_vpin_len'] = len(seq_data)
    return out


In [ ]:
def run_inference(sample_list, label):
    dataset = V3ContinuousDataset(
        sample_list,
        fused_tail_shape=(FUSED_SPATIAL_CHANNELS, FUSED_SPATIAL_BINS),
    )
    loader = DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        collate_fn=v3_collate_fn,
        num_workers=0,
    )

    pos_list, tgt_list = [], []
    model.eval()
    if hasattr(model, 'reset_position_state'):
        model.reset_position_state()
    with torch.no_grad():
        for batch in loader:
            inputs, targets = unpack_v3_batch(batch, device=device)
            out = model(**inputs, return_ae_losses=True)
            if USE_PTP:
                position, _ae_losses, _logits = out
            else:
                position, _ae_losses = out
            pos_list.append(position.view(-1).cpu().numpy())
            tgt_list.append(targets.view(-1).cpu().numpy())

    pos = np.concatenate(pos_list)
    tgt = np.concatenate(tgt_list)
    strat = pos * tgt
    return pd.DataFrame({
        'label': label,
        'position': pos,
        'target': tgt,
        'strategy_ret': strat,
    })

def summarize_case(label, df_case, base_df=None):
    sr = df_case['strategy_ret'].to_numpy()
    row = {
        'case': label,
        'n': len(df_case),
        'mean_abs_position': float(np.abs(df_case['position']).mean()),
        'gross_pnl': float(sr.sum()),
        'ann_sharpe': float(sr.mean() / (sr.std() + 1e-8) * np.sqrt(252 * (450 / TARGET_HORIZON_MINUTES))),
    }
    if base_df is not None:
        base_pos = base_df['position'].to_numpy()
        row['mean_abs_delta_vs_base'] = float(np.mean(np.abs(df_case['position'].to_numpy() - base_pos)))
        row['pos_corr_vs_base'] = float(np.corrcoef(df_case['position'].to_numpy(), base_pos)[0, 1])
    return row

eval_n = min(EVAL_MAX_SAMPLES, len(oos_samples))
eval_samples = oos_samples[:eval_n]
baseline_df = run_inference(eval_samples, 'baseline')
tech_shift_df = run_inference([perturb_tech_shift_one(s) for s in eval_samples], 'tech_shift_1')
seq_early_df = run_inference([perturb_seq_cutoff_earlier(s) for s in eval_samples], 'seq_minus_15m')

summary = pd.DataFrame([
    summarize_case('baseline', baseline_df),
    summarize_case('tech_shift_1', tech_shift_df, baseline_df),
    summarize_case('seq_minus_15m', seq_early_df, baseline_df),
])
print(summary.to_string(index=False))
